# Combined Contextual Reranking Model

This notebook implements a combined contextual reranking model for the Activity Recommendation System.

The model integrates five scoring components:
- Bayesian Personalized Ranking (BPR)
- Weather suitability
- Time availability
- Distance suitability
- Budget suitability

The evaluation scenario uses rainy weather on Saturday at 2:00 PM.

The implementation produces top-10 activity recommendations for each user and supports configurable scoring weights.

Weight sensitivity analysis, ablation experiments, and recommendation evaluation are conducted separately.

## 1. Setup and Data Loading

Load the shared BPR candidate recommendations and activity metadata required for contextual reranking.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project directory
project = Path.cwd().parent

# Load shared BPR candidates
candidates = pd.read_csv(
    project / "data/shared_bpr_candidates.csv"
)

# Load activity metadata
activities = pd.read_csv(
    project / "data/processed/activity_dataset.csv",
    low_memory=False
)

# Remove identical duplicate activity records
activities = activities.drop_duplicates(subset="gmap_id")

# Combine BPR candidates with activity metadata
combined_df = candidates.merge(
    activities,
    on="gmap_id",
    how="left",
    validate="many_to_one"
)

print("Total candidates:", len(combined_df))
print("Unique users:", combined_df["user_id"].nunique())
print("Missing activity records:", combined_df["name"].isna().sum())

Total candidates: 10000
Unique users: 50
Missing activity records: 0


## 2. Weather and Time Scoring

This section calculates contextual suitability scores for rainy weather on Saturday at 2:00 PM.

Weather suitability uses the final V3 activity exposure classifications. Time suitability determines whether each activity is open, closed, or has unknown availability.

Confirmed-closed activities will be excluded during recommendation generation.

In [2]:
# Load final V3 weather exposure classifications
weather_data = pd.read_csv(
    project / "data/processed/activities_weather_exposure_v3.csv",
    low_memory=False
)

# Scores for the agreed rainy-weather scenario
rain_scores = {
    "indoor": 1.0,
    "outdoor": 0.2,
    "mixed": 0.7,
    "unknown": 0.7
}

weather_data["weather_score"] = (
    weather_data["weather_exposure_final"].map(rain_scores)
)

# Add weather suitability to BPR candidates
combined_df = combined_df.merge(
    weather_data[["gmap_id", "weather_score"]],
    on="gmap_id",
    how="left",
    validate="many_to_one"
)

print("Candidates:", len(combined_df))
print("Missing weather scores:", combined_df["weather_score"].isna().sum())

Candidates: 10000
Missing weather scores: 0


In [3]:
# Load previously calculated time availability scores
time_data = pd.read_csv(
    project / "data/processed/activities_weather_exposure_v3.csv",
    low_memory=False
)

# Add time scores to the combined candidates
combined_df = combined_df.merge(
    time_data[["gmap_id", "time_availability", "time_score"]],
    on="gmap_id",
    how="left",
    validate="many_to_one"
)

print("Total candidates:", len(combined_df))
print("Missing time scores:", combined_df["time_score"].isna().sum())
print("\nTime availability:")
print(combined_df["time_availability"].value_counts())

Total candidates: 10000
Missing time scores: 0

Time availability:
time_availability
open       8503
closed      803
unknown     694
Name: count, dtype: int64


## 3. Distance and Budget Scoring

This section integrates distance and budget suitability into the combined recommendation model, using the scoring methods developed in the distance-versus-budget experiments.

Both scores will be placed on a consistent 0–1 scale for weighted reranking.

In [7]:
import sys

if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from distance import add_distance_scores
from budget import add_budget_scores

# Same assumptions used in 07_budget_vs_distance.ipynb experiments
user_location = (40.7580, -73.9855)
radius_miles = 5
max_budget = 2

combined_df = add_distance_scores(
    combined_df, user_location, radius_miles
)

combined_df = add_budget_scores(
    combined_df, max_budget
)

print("Candidates:", len(combined_df))
print("Missing distance scores:", combined_df["distance_score"].isna().sum())
print("Missing budget scores:", combined_df["budget_score"].isna().sum())

display(combined_df[
    ["gmap_id", "distance_miles", "distance_score",
     "price", "budget_score"]
].head())

Candidates: 10000
Missing distance scores: 0
Missing budget scores: 0


,gmap_id,distance_miles,distance_score,price,budget_score
0,0x89c2f3747f378f7d:0x61caa819e454f065,9.764754,-0.952951,NaN,0.0
1,0x89c2f4e9f911e9d9:0x63082da1140b9e66,8.078725,-0.615745,$$,1.0
2,0x89c2f3a32b7ceaeb:0x3c3cecb4c22cf890,9.799354,-0.959871,$$,1.0
3,0x89c2f3926691fd75:0xae683d2e5147a9a8,8.938753,-0.787751,$$,1.0
4,0x89c28cba56f51c3d:0xe19c58eeca84032f,11.036146,-1.000000,$$,1.0


In [9]:
import sys

# Make project modules accessible
if str(project) not in sys.path:
    sys.path.insert(0, str(project))

from distance import add_distance_scores
from budget import add_budget_scores

# Scenario settings from previous experiments
user_location = (40.7580, -73.9855)  # Times Square, NYC
radius_miles = 5
max_budget = 2  # $$

# Calculate distance suitability
combined_df = add_distance_scores(
    combined_df,
    user_location,
    radius_miles
)

# Calculate budget suitability
combined_df = add_budget_scores(
    combined_df,
    max_budget
)

# Verify scores
print("Total candidates:", len(combined_df))
print("Missing distance scores:", combined_df["distance_score"].isna().sum())
print("Missing budget scores:", combined_df["budget_score"].isna().sum())

print("\nDistance score range:",
      combined_df["distance_score"].min(),
      "to",
      combined_df["distance_score"].max())

print("Budget score range:",
      combined_df["budget_score"].min(),
      "to",
      combined_df["budget_score"].max())

Total candidates: 10000
Missing distance scores: 0
Missing budget scores: 0

Distance score range: -1.0 to 0.9968178715238613
Budget score range: -1.0 to 1.0


## 4. Combined Contextual Reranking

This section combines BPR relevance, weather suitability, time availability, distance suitability, and budget suitability into a single weighted recommendation score.

Negative distance and budget penalties are preserved. The function accepts configurable weights that must sum to 1, allowing subsequent weight sensitivity analysis.

In [10]:
def combined_rerank(
    df,
    weights=(0.40, 0.15, 0.15, 0.15, 0.15)
):
    """
    Calculate combined contextual recommendation scores.

    Weight order:
    BPR, weather, time, distance, budget.
    """
    weights = np.asarray(weights, dtype=float)

    if len(weights) != 5 or np.any(weights < 0):
        raise ValueError("Provide five nonnegative weights.")

    if not np.isclose(weights.sum(), 1.0):
        raise ValueError("Weights must sum to 1.")

    score_columns = [
        "bpr_normalized",
        "weather_score",
        "time_score",
        "distance_score",
        "budget_score"
    ]

    result = df.copy()

    if result[score_columns].isna().any().any():
        raise ValueError("Missing component scores.")

    result["combined_score"] = (
        result[score_columns].to_numpy() @ weights
    )

    return result.sort_values(
        ["user_id", "combined_score"],
        ascending=[True, False]
    ).reset_index(drop=True)


# Initial weights for implementation testing
ranked_df = combined_rerank(combined_df)

print("Total ranked candidates:", len(ranked_df))
print(
    "Combined score range:",
    ranked_df["combined_score"].min(),
    "to",
    ranked_df["combined_score"].max()
)

Total ranked candidates: 10000
Combined score range: -0.15 to 0.992510343114288


## 5. Top-10 Recommendation Generation

This section excludes activities confirmed to be closed and selects the 10 highest-scoring eligible activities for each user.

Activities with unknown opening hours remain eligible. Recommendations are ranked by their combined contextual scores.

In [12]:
def generate_top10(ranked_df, k=10):
    """
    Exclude confirmed-closed activities and return
    the top-k recommendations for each user.
    """
    eligible = ranked_df[
        ranked_df["time_availability"].str.lower().ne("closed")
    ].copy()

    eligible = eligible.sort_values(
        ["user_id", "combined_score", "gmap_id"],
        ascending=[True, False, True]
    )

    top10 = eligible.groupby(
        "user_id", group_keys=False
    ).head(k).copy()

    top10["recommendation_rank"] = (
        top10.groupby("user_id").cumcount() + 1
    )

    return top10.reset_index(drop=True)


top10_df = generate_top10(ranked_df)

print("Total recommendations:", len(top10_df))
print("Users:", top10_df["user_id"].nunique())
print("Closed recommendations:",
      top10_df["time_availability"].str.lower().eq("closed").sum())

display(top10_df[
    ["user_id", "recommendation_rank", "name",
     "combined_score", "time_availability"]
].head(10))

Total recommendations: 500
Users: 50
Closed recommendations: 0


,user_id,recommendation_rank,name,combined_score,time_availability
0,100388958952721979686,1,Sabrosura 2,0.732676,open
1,100388958952721979686,2,Hibachi Grill & Supreme Buffet,0.689857,open
2,100388958952721979686,3,Applebee's Grill + Bar,0.674815,open
3,100388958952721979686,4,Buffalo Wild Wings,0.663627,open
4,100388958952721979686,5,White Castle,0.658002,open
5,100388958952721979686,6,Dallas BBQ Fordham Road,0.650661,open
6,100388958952721979686,7,McDonald's,0.639872,open
7,100388958952721979686,8,Outback Steakhouse,0.634793,open
8,100388958952721979686,9,White Castle.,0.632909,open
9,100388958952721979686,10,Wendy's,0.619335,open


## 6. Implementation Validation and Export

This section verifies the final recommendation output and exports the scored candidates and top-10 recommendations for subsequent evaluation.

Weight optimization, ablation studies, and recommendation metrics are handled in the separate evaluation notebook.

In [13]:
# Validate recommendation output
assert len(top10_df) == 500
assert top10_df["user_id"].nunique() == 50
assert top10_df.groupby("user_id").size().eq(10).all()
assert top10_df["gmap_id"].notna().all()
assert not top10_df.duplicated(["user_id", "gmap_id"]).any()
assert not top10_df["time_availability"].str.lower().eq("closed").any()

# Save full scored candidates and final top-10 recommendations
output_dir = project / "data/processed"

ranked_df.to_csv(
    output_dir / "combined_contextual_candidates.csv",
    index=False
)

top10_df.to_csv(
    output_dir / "combined_contextual_top10.csv",
    index=False
)

print("Validation passed.")
print("Saved combined_contextual_candidates.csv")
print("Saved combined_contextual_top10.csv")

Validation passed.
Saved combined_contextual_candidates.csv
Saved combined_contextual_top10.csv
